In [ ]:
# import required libraries
import cv2
import face_recognition
import os
import numpy as np

def getImagesAndFileName():
    images = []
    imageFileNames = []
    path = r'C:\Users\Sourabha\Documents\Pictures'
    # get all images and file names in path and store in the lists
    fileNames = os.listdir(path)
    for filename in fileNames:
        # read image with OpenCV's imread
        imageFile = cv2.imread(f'{path}/{filename}')
        images.append(imageFile)
        # get image file name without image type
        imageFileName = os.path.splitext(filename)[0]
        imageFileNames.append(imageFileName) 
    return images, imageFileNames


# get encodings of the images from the path specified
def getEncodings(images):
    faceEncodings = []
    for img in images:
        # convert the image to RGB format
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        encode = face_recognition.face_encodings(img)[0]
        faceEncodings.append(encode)
    return faceEncodings


def faceDetection():
    # Image resizing and color conversion
    imageScaled = cv2.resize(frame, (0,0), None, 0.3, 0.3)
    imageScaled = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    # Get face locations in the video frame
    frameFaceLocation = face_recognition.face_locations(imageScaled)  
    # Get face encodings in the video frame
    frameFaceEncoding = face_recognition.face_encodings(imageScaled, frameFaceLocation)  
    # Compare face in video frame and from existing folder path
    for faceEncoding, faceLocation in zip(frameFaceEncoding, frameFaceLocation):
        # using face_recognition library to compare faces
        compareImages = face_recognition.compare_faces(existingImagesEncoding, faceEncoding)
        faceDistance = face_recognition.face_distance(existingImagesEncoding, faceEncoding)
        # get mininmum value from faceDistance
        matchIndex = np.argmin(faceDistance)
        # if the face on video frame matches the existing images from folder
        if compareImages[matchIndex]:     
            name = imageFileNames[matchIndex].upper()
            top, right, bottom, left = faceLocation
            # Draw a rectangle around the detected face in video frame
            cv2.rectangle(frame, (left, top), (right, bottom), (255, 0, 0), 2)
            # Draw a filled rectangle with text background in video frame
            cv2.rectangle(frame, (left, bottom+30), (right, bottom), (255, 0, 0), cv2.FILLED) 
            cv2.putText(frame, name, (left+15, bottom+25), cv2.FONT_HERSHEY_PLAIN, 0.8, (255,255,255), 2)

            
images, imageFileNames = getImagesAndFileName()
print('Saved images and the file names for face detection.')
existingImagesEncoding = getEncodings(images)
print('Encoding Complete!')

# Open webcam
videoCapture = cv2.VideoCapture(0)
while videoCapture.isOpened():
    ret, frame = videoCapture.read()
    faceDetection()
    cv2.imshow('Webcam', frame)
    
    # press 'ESC' to quit
    if cv2.waitKey(1) == 27: 
        break

videoCapture.release()
cv2.destroyAllWindows()
print('Closed!')